# **COMP 3610 - Assignment 4: MLOps & Model Deployment**

**Name:** Samuel Soman  
**ID:** 816039318 

### **Project Overview**

This project focuses on deploying a trained machine learning model as a complete prediction service. Using the NYC Yellow Taxi Trip Records dataset and a regression model developed in Assignment 2, the goal is to predict `tip_amount` based on trip-related features such as distance, time, and fare details.
The project includes several key stages of the machine learning deployment pipeline. First, MLflow is used to track experiments, compare model performance, and manage model versions through the Model Registry. Next, the selected best-performing regression model is exposed through a REST API built with FastAPI, allowing users to send input data and receive predicted tip amounts. Proper input validation and error handling are also included to make the service reliable.
To make the application portable and easy to run, the entire system is containerized using Docker and Docker Compose. This allows all components to be started with a single command and makes the service suitable for deployment on different environments or cloud platforms.
Overall, this project demonstrates an end-to-end machine learning deployment workflow, moving from a trained model to a containerized, API-accessible prediction service.

This notebook documents the end-to-end process of deploying a trained NYC Yellow Taxi tip-prediction model as a containerised REST API, covering:

| Part | Topic | Weight |
|------|-------|--------|
| 1 | Experiment Tracking with MLflow | 25 % |
| 2 | Model Serving with FastAPI | 35 % |
| 3 | Containerization with Docker | 20 % |
| 4 | Documentation & Code Quality | 10 % |

---

### <u>**0.0 Setup & Data Preparation**</u>

We begin by installing dependencies that are already listed in `requirements.txt`, downloading the NYC taxi dataset, and engineering the same features used in Assignment 2.

In [1]:
# Install dependencies (safe to re-run)
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")

NumPy  : 2.2.6
Pandas : 2.2.3


### <u>**0.1 Download the Dataset**</u>

We use the **Yellow Taxi Trip Data (January 2024)** parquet file.  
The file is downloaded only if it is not already present.

In [3]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TAXI_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
TAXI_FILE = DATA_DIR / "yellow_tripdata_2024-01.parquet"

if not TAXI_FILE.exists():
    print("Downloading taxi trip data …")
    import urllib.request
    urllib.request.urlretrieve(TAXI_URL, TAXI_FILE)
    print(f"Saved to {TAXI_FILE}")
else:
    print(f"Data already present at {TAXI_FILE}")

df = pd.read_parquet(TAXI_FILE)
print(f"Raw dataset shape: {df.shape}")

Data already present at data\yellow_tripdata_2024-01.parquet
Raw dataset shape: (2964624, 19)


### <u>**0.2 Data Cleaning & Feature Engineering**</u>

The same cleaning rules and feature derivations from Assignment 2 are applied here.  
Borough one-hot features are **omitted** to keep the deployment API simpler without meaningfully hurting Random Forest performance (borough importances < 0.002 each in Assignment 2).

In [4]:
# ---------- cleaning ----------
df = df[df["payment_type"] == 1].copy()          # credit-card only
df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 200)]
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
df = df[(df["tip_amount"] >= 0) & (df["tip_amount"] < 100)]
df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 9)]

# ---------- temporal features ----------
df["pickup_hour"]        = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek

# ---------- trip duration ----------
df["trip_duration_minutes"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
)
df = df[(df["trip_duration_minutes"] > 0) & (df["trip_duration_minutes"] < 180)]

# ---------- derived features ----------
df["trip_speed_mph"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["trip_distance"] / (df["trip_duration_minutes"] / 60),
    0,
)
df["trip_speed_mph"] = df["trip_speed_mph"].clip(upper=80)

df["log_trip_distance"] = np.log1p(df["trip_distance"])

df["fare_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["fare_amount"] / df["trip_distance"],
    0,
)
df["fare_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["fare_amount"] / df["trip_duration_minutes"],
    0,
)

df["is_weekend"] = (df["pickup_day_of_week"] >= 5).astype(int)

for col in ["congestion_surcharge", "Airport_fee"]:
    df[col] = df[col].fillna(0)

In [5]:
# ---------- feature matrix ----------
FEATURE_COLS = [
    "passenger_count", "trip_distance", "RatecodeID",
    "fare_amount", "extra", "mta_tax", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "Airport_fee",
    "pickup_hour", "pickup_day_of_week",
    "trip_duration_minutes", "trip_speed_mph",
    "log_trip_distance", "fare_per_mile", "fare_per_minute",
    "is_weekend",
]

X = df[FEATURE_COLS].copy()
y = df["tip_amount"].copy()

# Fill residual NaN with median
X = X.fillna(X.median())

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")


Features : 18
Samples  : 2,271,426


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")

Train : 1,817,140  |  Test : 454,286


---
# **Part 1: Experiment Tracking with MLflow**
---

We use a **local SQLite-backed MLflow tracking store** (`sqlite:///mlflow.db`) so the Model Registry is available without needing a separate server process. All experiment data is persisted to `mlflow.db` and artifacts to the local `mlartifacts/` directory.

To browse the runs in the MLflow UI, you can optionally start the tracking UI in a separate terminal:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Then open <http://localhost:5000> in a browser.

In [7]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Use a local SQLite-backed tracking store so the Model Registry works
# without needing a separate MLflow server process running.
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# Create (or reuse) the experiment
mlflow.set_experiment("taxi-tip-prediction")

print(f"MLflow version : {mlflow.__version__}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")
print(f"Experiment     : taxi-tip-prediction")

2026/04/17 18:47:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/17 18:47:22 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step
INFO  [alembic.runtime.migration] Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
INFO  [alembic.runtime.migration] Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
INFO  [alembic.runtime.migration] Running upgrade 181f10493468 -> df50e92ffc5e, Add Experiment Tags Table
INFO  [alembic.runtime.migration] Running upgrade df50e92ffc5e -> 7ac759974ad8, Update run tags with larger limit
INFO  [alembic.runtime.migration] Running upgrade 7ac759974ad8 -> 89d4b8295536, create latest metrics table
INFO  [89d4b8295536_create_latest_metrics_table_py] Migration complete!
INFO  

MLflow version : 2.22.0
Tracking URI   : sqlite:///mlflow.db
Experiment     : taxi-tip-prediction


In [8]:
def log_regression_metrics(y_true, y_pred):
    """Compute and log MAE, RMSE, R² to the active MLflow run."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = r2_score(y_true, y_pred)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

### <u>**Model 1 - Random Forest Regressor (baseline)**</u>
Train a Random Forest with default-style hyperparameters and log everything to MLflow.

In [9]:
with mlflow.start_run(run_name="rf-baseline"):
    # Parameters
    params = {
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "random_state": RANDOM_STATE,
    }
    mlflow.log_params(params)

    # Train
    rf_baseline = RandomForestRegressor(**params, n_jobs=-1)
    rf_baseline.fit(X_train, y_train)

    # Evaluate & log metrics
    preds = rf_baseline.predict(X_test)
    metrics_baseline = log_regression_metrics(y_test, preds)

    # Tags
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("dataset_version", "yellow_tripdata_2024-01")
    mlflow.set_tag("author", "Samuel Soman")

    # Log model artifact
    mlflow.sklearn.log_model(rf_baseline, "model")

    baseline_run_id = mlflow.active_run().info.run_id

print(f"RF Baseline  — run_id: {baseline_run_id}")
for k, v in metrics_baseline.items():
    print(f"  {k}: {v:.4f}")

2026/04/17 18:55:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RF Baseline  — run_id: 077820e522c74201b1d89a574881601d
  MAE: 1.1883
  RMSE: 2.2583
  R2: 0.6418


### <u>**Model 2 - Random Forest Regressor (tuned)</u>**

This model uses the best hyperparameters found in Assignment 2 via `RandomizedSearchCV`.

In [10]:
with mlflow.start_run(run_name="rf-tuned"):
    # Best hyperparameters from Assignment 2 tuning
    params = {
        "n_estimators": 166,
        "max_depth": 10,
        "min_samples_split": 6,
        "min_samples_leaf": 2,
        "max_features": None,
        "random_state": RANDOM_STATE,
    }
    mlflow.log_params(params)

    # Train
    rf_tuned = RandomForestRegressor(**params, n_jobs=-1)
    rf_tuned.fit(X_train, y_train)

    # Evaluate & log metrics
    preds_tuned = rf_tuned.predict(X_test)
    metrics_tuned = log_regression_metrics(y_test, preds_tuned)

    # Tags
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("dataset_version", "yellow_tripdata_2024-01")
    mlflow.set_tag("author", "Samuel Soman")
    mlflow.set_tag("tuning", "RandomizedSearchCV_best")

    # Log model artifact
    mlflow.sklearn.log_model(rf_tuned, "model")

    tuned_run_id = mlflow.active_run().info.run_id

print(f"RF Tuned     — run_id: {tuned_run_id}")
for k, v in metrics_tuned.items():
    print(f"  {k}: {v:.4f}")

2026/04/17 19:03:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


RF Tuned     — run_id: 6670cbae74d747b9a7b44710a51dda58
  MAE: 1.1818
  RMSE: 2.2344
  R2: 0.6493


### <u>**Model 3 - Linear Regression</u>**

A simple linear baseline to compare against the tree-based models (following the pattern in the MLflow Quickstart guide).

In [11]:
with mlflow.start_run(run_name="linear-regression"):
    mlflow.log_params({"model_type": "LinearRegression"})

    lr = LinearRegression()
    lr.fit(X_train, y_train)

    preds_lr = lr.predict(X_test)
    metrics_lr = log_regression_metrics(y_test, preds_lr)

    # Tags
    mlflow.set_tag("model_type", "LinearRegression")
    mlflow.set_tag("dataset_version", "yellow_tripdata_2024-01")
    mlflow.set_tag("author", "Samuel Soman")

    # Log model artifact
    mlflow.sklearn.log_model(lr, "model")

    lr_run_id = mlflow.active_run().info.run_id

print(f"Linear Reg   — run_id: {lr_run_id}")
for k, v in metrics_lr.items():
    print(f"  {k}: {v:.4f}")

2026/04/17 19:03:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Reg   — run_id: 72b5984617dc4d648fe7a93d368fcaae
  MAE: 1.2105
  RMSE: 2.2643
  R2: 0.6398


### **MLflow UI Evidence**

> **Action:** open <http://localhost:5000> → navigate to the *taxi-tip-prediction* experiment → verify all three runs are visible with parameters, metrics, and tags.

![mlflow_ui_runs.png](images/mlflow_ui_runs.png)

The screenshot above shows the MLflow UI with all three logged runs — `rf-baseline`, `rf-tuned`, and `linear-regression` — under the `taxi-tip-prediction` experiment. Each run completed successfully (green status), and the MAE, R², and RMSE metrics are visible in the table columns, confirming that parameters, metrics, and model artifacts were logged correctly for every run.

---
### **Task 1.2 - Model Comparison & Registry**
---

### Programmatic comparison of logged runs

We also retrieve and compare runs via the MLflow API.

In [12]:
experiment = mlflow.get_experiment_by_name("taxi-tip-prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

comparison_cols = [
    "run_id", "tags.mlflow.runName",
    "metrics.mae", "metrics.rmse", "metrics.r2",
]
print("Side-by-side comparison of all logged runs:\n")
runs[comparison_cols].sort_values("metrics.r2", ascending=False)

Side-by-side comparison of all logged runs:



,run_id,tags.mlflow.runName,metrics.mae,metrics.rmse,metrics.r2
1,6670cbae74d747b9a7b44710a51dda58,rf-tuned,1.181765,2.234450,0.649278
2,077820e522c74201b1d89a574881601d,rf-baseline,1.188289,2.258271,0.641760
0,72b5984617dc4d648fe7a93d368fcaae,linear-regression,1.210467,2.264316,0.639840


### **Side-by-side comparison screenshot**

> **Action:** in the MLflow UI, select all three runs → click **Compare** → take a screenshot of the comparison view.

![mlflow_comparison_1.png](images/mlflow_comparison_1.png)
![mlflow_comparison_2.png](images/mlflow_comparison_2.png)

The comparison view confirms that the **tuned Random Forest** (`rf-tuned`) is the best-performing model across all three metrics — it has the lowest MAE (1.18), the lowest RMSE (2.23), and the highest R² (0.649). The baseline RF with `max_depth=20` performs slightly worse, while Linear Regression trails both tree-based models. This justifies selecting `rf-tuned` for registration and deployment.

### <u>**Best Model Analysis**</u>

Three models were compared:

| Model | Description |
|-------|-------------|
| **RF Baseline** | Random Forest with default parameters |
| **RF Tuned** | Random Forest with hyperparameters from Assignment 2's `RandomizedSearchCV` |
| **Linear Regression** | Simple linear model as a non-tree baseline |

The **tuned Random Forest** outperforms the other two across all three metrics:

- **Lower MAE** - the tuned model's mean absolute error is smallest, meaning its average per-trip tip prediction is closest to the true value.
- **Lower RMSE** - fewer extreme mispredictions compared to the baseline RF and Linear Regression.
- **Higher R²** - a larger proportion of tip variance is explained, confirming best overall fit.

Linear Regression provides a useful lower bound - it captures the linear relationship between fare and tip but cannot model the non-linear interactions that the Random Forest captures (e.g., interaction between trip distance and time of day).

The improvement of the tuned RF over the baseline RF stems from the hyperparameter search conducted in Assignment 2, which found optimal tree depth and leaf constraints that reduce over-fitting on the training set while retaining predictive signal.

### <u>**Register the best model in the MLflow Model Registry**</u>

In [13]:
MODEL_REGISTRY_NAME = "taxi-tip-regressor"

# Register the tuned model from its run
model_uri = f"runs:/{tuned_run_id}/model"
reg_result = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)

print(f"Registered model : {reg_result.name}")
print(f"Version          : {reg_result.version}")

Registered model : taxi-tip-regressor
Version          : 1


Successfully registered model 'taxi-tip-regressor'.
Created version '1' of model 'taxi-tip-regressor'.


In [14]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Add a version description documenting its performance
client.update_model_version(
    name=MODEL_REGISTRY_NAME,
    version=reg_result.version,
    description=(
        f"Tuned Random Forest Regressor (n_estimators=166, max_depth=10). "
        f"MAE={metrics_tuned['MAE']:.4f}, RMSE={metrics_tuned['RMSE']:.4f}, "
        f"R²={metrics_tuned['R2']:.4f}. Trained on NYC yellow_tripdata_2024-01."
    ),
)
print("Version description updated.")

Version description updated.


### <u>**Set model alias for deployment**</u>

We assign the `champion` alias to the best model version so the API can reference it by alias. This replaces the deprecated Stage-based workflow in newer MLflow versions.

In [15]:
# Set the "champion" alias on the best model version
client.set_registered_model_alias(
    name=MODEL_REGISTRY_NAME,
    alias="champion",
    version=reg_result.version,
)
print(f"Model '{MODEL_REGISTRY_NAME}' version {reg_result.version} → alias 'champion'")

Model 'taxi-tip-regressor' version 1 → alias 'champion'


### <u>**Load model from registry & make a sample prediction**</u>

In [16]:
# Load the champion model from the Model Registry using alias
loaded_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_REGISTRY_NAME}@champion"
)

# Use a single sample from the test set
sample = X_test.iloc[[0]]
actual  = y_test.iloc[0]
predicted = loaded_model.predict(sample)[0]

print(f"Sample prediction from registry model (champion alias):")
print(f"  Actual tip  : ${actual:.2f}")
print(f"  Predicted   : ${predicted:.2f}")

Sample prediction from registry model (champion alias):
  Actual tip  : $11.84
  Predicted   : $10.97


### <u>**Save model for API deployment**</u>

Export the best model as a `joblib` file so the FastAPI app can load it without a running MLflow server.

In [17]:
import joblib

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "rf_regressor.joblib"

joblib.dump(rf_tuned, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")

Model saved to models\rf_regressor.joblib  (12984 KB)


---
# **Part 2: Model Serving with FastAPI** 
---

### <u>**Task 2.1 - API Design & Implementation**</u>

The full API lives in **`app.py`**.  Key design decisions:

1. **Model loaded once at startup** via FastAPI's `lifespan` context manager - never reloaded per-request.
2. **Pydantic `TripInput` model** validates **13 input fields** (5+ with constraints): `trip_distance` (gt=0), `passenger_count` (ge=1, le=9), `fare_amount` (ge=0), `pickup_hour` (ge=0, le=23), `pickup_day_of_week` (ge=0, le=6), `trip_duration_minutes` (ge=0), `RatecodeID` (ge=1, le=6), etc.
3. **Derived features** (`trip_speed_mph`, `log_trip_distance`, `fare_per_mile`, `fare_per_minute`, `is_weekend`) are computed server-side in `compute_features()` so the caller only provides raw trip attributes.
4. **Response** includes `tip_amount` (rounded to 2 d.p.), `model_version`, and a UUID `prediction_id`.

Below is the complete `app.py` source:

In [18]:
# Display app.py source for review
print(open("app.py").read())

"""
FastAPI application for NYC Taxi Tip Prediction Service.

Serves tip_amount predictions from a trained Random Forest model
using the NYC Yellow Taxi Trip Records dataset.
"""

import uuid
import os
import time
import logging
from contextlib import asynccontextmanager

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List


# ---------------------------------------------------------------------------
# Configuration (environment variables with sensible defaults)
# ---------------------------------------------------------------------------
MODEL_PATH = os.getenv("MODEL_PATH", "models/rf_regressor.joblib")
MODEL_VERSION = os.getenv("MODEL_VERSION", "1")
MODEL_NAME = "taxi-tip-regressor"

# Logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("tip-prediction-api")

# Global references â€“ populated at startup
ml_model = None
s

### **<u>Task 2.2 — Enhanced API Features</u>**

The following endpoints extend the base API (all implemented inside `app.py`):

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/predict/batch` | POST | Accepts a list of up to **100** trip records; enforced via `max_length=100` on the Pydantic model. |
| `/health` | GET | Returns API status, `model_loaded` flag, `model_version`, and `uptime_seconds`. |
| `/model/info` | GET | Returns metadata: model name, version, feature names, and training metrics (MAE, RMSE, R²). |
| *(global)* | — | A `@app.exception_handler(Exception)` catches unexpected errors and returns a structured `{"error": ..., "detail": ...}` JSON body with HTTP 500, **without** exposing internal stack traces. |

### <u>**Task 2.3 - API Testing</u>**

Automated tests are defined in **`test_app.py`** using `pytest` and FastAPI's `TestClient`.  
Below is the test source and execution output.

In [19]:
# Display test_app.py source for review
print(open("test_app.py").read())

"""
Test suite for the NYC Taxi Tip Prediction API.

Run with:  pytest test_app.py -v
"""

import pytest
from fastapi.testclient import TestClient
from app import app


@pytest.fixture(scope="module")
def client():
    """Create a TestClient with lifespan events (model loading)."""
    with TestClient(app) as c:
        yield c

# Reference payload with valid values for reuse across tests
VALID_TRIP = {
    "trip_distance": 3.5,
    "passenger_count": 1,
    "fare_amount": 15.0,
    "pickup_hour": 14,
    "pickup_day_of_week": 2,
    "trip_duration_minutes": 12.0,
    "RatecodeID": 1,
    "extra": 1.0,
    "mta_tax": 0.5,
    "tolls_amount": 0.0,
    "improvement_surcharge": 1.0,
    "congestion_surcharge": 2.5,
    "Airport_fee": 0.0,
}


# ---- Test 1: Successful single prediction ---------------------------------
def test_single_prediction(client):
    """Valid input should return 200 with tip_amount, prediction_id, model_version."""
    response = client.post("/predict", json=VALID

In [20]:
# Run the test suite from the notebook
!pytest test_app.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.7, pytest-8.3.5, pluggy-1.6.0 -- c:\Users\Samuel Soman\COMP3610-A4\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Samuel Soman\COMP3610-A4
plugins: anyio-4.13.0
collecting ... collected 10 items

test_app.py::test_single_prediction PASSED                               [ 10%]
test_app.py::test_batch_prediction PASSED                                [ 20%]
test_app.py::test_invalid_input_missing_fields PASSED                    [ 30%]
test_app.py::test_invalid_input_out_of_range PASSED                      [ 40%]
test_app.py::test_health_check PASSED                                    [ 50%]
test_app.py::test_zero_distance_trip PASSED                              [ 60%]
test_app.py::test_extreme_fare_values PASSED                             [ 70%]
test_app.py::test_model_info PASSED                                      [ 80%]
test_app.py::test_batch_exceeds_ma

### Swagger UI Documentation

Start the server in a terminal with:

```bash
uvicorn app:app --reload --port 8000
```

Then visit <http://localhost:8000/docs> for the auto-generated Swagger UI.

> **Action:** take a screenshot of the Swagger UI at `/docs` and paste it here.

![docker_evidence_1.png](images/docker_evidence_1.png)
![docker_evidence_2.png](images/docker_evidence_2.png)


---
# **Part 3: Containerization with Docker** 
---

### **<u>Task 3.1 - Dockerfile & Image Building</u>**

The `Dockerfile` uses `python:3.11-slim` and follows best practices:
- Copies `requirements.txt` first for layer caching.
- Installs dependencies with `--no-cache-dir` for a smaller image.
- Copies only `app.py` and `models/`.
- Exposes port 8000 and starts `uvicorn`.

A `.dockerignore` excludes data files, notebooks, `__pycache__/`, `.git/`, etc.

In [21]:
# Show Dockerfile
print(open("Dockerfile").read())

# Dockerfile for NYC Taxi Tip Prediction API
# Uses a slim Python base image to keep the final image small.

# 1. Base image
FROM python:3.11-slim

# 2. Working directory inside the container
WORKDIR /app

# 3. Copy dependency file first (layer caching optimisation)
COPY requirements.txt .

# 4. Install Python dependencies (no pip cache â†’ smaller image)
RUN pip install --no-cache-dir -r requirements.txt

# 5. Copy application code and model artifacts
COPY app.py .
COPY models/ ./models/

# 6. Expose the port uvicorn will listen on
EXPOSE 8000

# 7. Start the API server
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



In [22]:
# Show .dockerignore
print(open(".dockerignore").read())

# Ignore data files
data/
*.parquet
*.csv

# Ignore MLflow tracking data
mlruns/
mlartifacts/

# Ignore Python cache
__pycache__/
*.pyc
*.pyo

# Ignore Git data
.git/
.gitignore

# Ignore notebooks and docs
*.ipynb
*.md
*.pdf

# Ignore test files
test_app.py
.pytest_cache/

# Ignore virtual environments
.venv/
venv/

# Ignore IDE files
.vscode/
.idea/

# Ignore OS files
.DS_Store
Thumbs.db

# Ignore environment files
.env



### <u>**Build the Docker image**</u>

In [23]:
!docker build -t taxi-tip-api .

#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 707B 0.0s done
#1 DONE 0.1s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 ...

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.11-slim
#2 DONE 1.0s

#4 [internal] load .dockerignore
#4 transferring context: 509B done
#4 DONE 0.0s

#5 [internal] load build context
#5 ...

#6 [1/6] FROM docker.io/library/python:3.11-slim@sha256:233de06753d30d120b1a3ce359d8d3be8bda78524cd8f520c99883bfe33964cf
#6 resolve docker.io/library/python:3.11-slim@sha256:233de06753d30d120b1a3ce359d8d3be8bda78524cd8f520c99883bfe33964cf 0.1s done
#6 DONE 0.1s

#5 [internal] load build context
#5 transferring context: 13.30MB 1.1s done
#5 DONE 1.2s

#7 [2/6] WORKDIR /app
#7 CACHED

#8 [3/6] COPY requirements.txt .
#8 CACHED

#9 [4/6] RUN pip install --no-cache-dir -r r

### <u>**Report image size**</u>

In [24]:
!docker images taxi-tip-api

IMAGE                 ID             DISK USAGE   CONTENT SIZE   EXTRA
taxi-tip-api:latest   b22f06037c48        1.3GB          294MB        


### <u>**Run the container and verify accessibility**</u>

In [25]:
# Start container in background
!docker run -d --name taxi-api-test -p 8000:8000 taxi-tip-api

717a29fca8e079c0b0e877c8cd00c83420231103d5cad0249ecbe24d33b2e1ec


In [26]:
import time, requests

# Wait for container to start
time.sleep(5)

# Test prediction from outside the container
payload = {
    "trip_distance": 3.5,
    "passenger_count": 1,
    "fare_amount": 15.0,
    "pickup_hour": 14,
    "pickup_day_of_week": 2,
    "trip_duration_minutes": 12.0,
}

resp = requests.post("http://localhost:8000/predict", json=payload)
print(f"Status : {resp.status_code}")
print(f"Body   : {resp.json()}")

Status : 200
Body   : {'prediction_id': 'bccb9a96-5550-45c2-9590-95dca338dc94', 'tip_amount': 3.32, 'model_version': '1'}


In [27]:
# Clean up standalone container
!docker stop taxi-api-test
!docker rm taxi-api-test

taxi-api-test
taxi-api-test


---
### **Task 3.2 - Docker Compose & Deployment Demo**

The `docker-compose.yml` defines two services:

| Service | Image | Port | Purpose |
|---------|-------|------|---------|
| `api` | Built from `Dockerfile` | 8000 | Prediction API |
| `mlflow` *(bonus)* | `python:3.11-slim` + mlflow | 5000 | MLflow tracking server |

Docker Compose creates a shared network so the API can reach MLflow at `http://mlflow:5000`.

In [28]:
# Show docker-compose.yml
print(open("docker-compose.yml").read())

# docker-compose.yml â€“ Orchestrates the prediction API and MLflow services.
# Start everything:   docker compose up --build
# Shut down:          docker compose down

services:
  # ---------- Prediction API ----------
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=models/rf_regressor.joblib
      - MODEL_VERSION=1
      - MLFLOW_TRACKING_URI=http://mlflow:5000
    depends_on:
      - mlflow
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3

  # ---------- MLflow Tracking Server (Bonus) ----------
  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.22.0
    ports:
      - "5000:5000"
    command: mlflow server --host 0.0.0.0 --port 5000
    volumes:
      - mlflow-data:/mlflow

volumes:
  mlflow-data:



### <u>**Start services with `docker compose up`**</u>

In [29]:
!docker compose up -d --build

#1 [internal] load local bake definitions
#1 reading from stdin 521B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 707B 0.0s done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 DONE 0.4s

#4 [internal] load .dockerignore
#4 transferring context: 509B done
#4 DONE 0.0s

#5 [internal] load build context
#5 transferring context: 138B 0.0s done
#5 DONE 0.0s

#6 [1/6] FROM docker.io/library/python:3.11-slim@sha256:233de06753d30d120b1a3ce359d8d3be8bda78524cd8f520c99883bfe33964cf
#6 resolve docker.io/library/python:3.11-slim@sha256:233de06753d30d120b1a3ce359d8d3be8bda78524cd8f520c99883bfe33964cf 0.1s done
#6 DONE 0.1s

#7 [2/6] WORKDIR /app
#7 CACHED

#8 [3/6] COPY requirements.txt .
#8 CACHED

#9 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#9 CACHED

#10 [5/6] COPY app.py .
#10 CACHED

#11 [6/6] COPY models/ ./models/
#11 CACHED

#12 exporting to image
#12 exporting layers done
#12 exportin

 Image comp3610-a4-api Building 
 Image comp3610-a4-api Built 
 Network comp3610-a4_default Creating 
 Network comp3610-a4_default Created 
 Container comp3610-a4-mlflow-1 Creating 
 Container comp3610-a4-mlflow-1 Created 
 Container comp3610-a4-api-1 Creating 
 Container comp3610-a4-api-1 Created 
 Container comp3610-a4-mlflow-1 Starting 
 Container comp3610-a4-mlflow-1 Started 
 Container comp3610-a4-api-1 Starting 
 Container comp3610-a4-api-1 Started 


In [30]:
!docker compose ps

NAME                   IMAGE                           COMMAND                  SERVICE   CREATED         STATUS                            PORTS
comp3610-a4-api-1      comp3610-a4-api                 "uvicorn app:app --h…"   api       6 seconds ago   Up 4 seconds (health: starting)   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
comp3610-a4-mlflow-1   ghcr.io/mlflow/mlflow:v2.22.0   "mlflow server --hos…"   mlflow    6 seconds ago   Up 5 seconds                      0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp


### <u>**Make at least 3 prediction requests to the containerised API</u>**

In [31]:
import time, requests

# Wait for services to start
time.sleep(10)

BASE = "http://localhost:8000"

# --- Request 1: Health check ---
r1 = requests.get(f"{BASE}/health")
print("Request 1 — GET /health")
print(f"  Status: {r1.status_code}  Body: {r1.json()}\n")

# --- Request 2: Single prediction ---
trip_a = {
    "trip_distance": 2.1,
    "passenger_count": 2,
    "fare_amount": 10.5,
    "pickup_hour": 8,
    "pickup_day_of_week": 0,
    "trip_duration_minutes": 9.0,
}
r2 = requests.post(f"{BASE}/predict", json=trip_a)
print("Request 2 — POST /predict (morning commute)")
print(f"  Status: {r2.status_code}  Body: {r2.json()}\n")

# --- Request 3: Single prediction (different scenario) ---
trip_b = {
    "trip_distance": 18.0,
    "passenger_count": 1,
    "fare_amount": 52.0,
    "pickup_hour": 22,
    "pickup_day_of_week": 5,
    "trip_duration_minutes": 35.0,
    "tolls_amount": 6.55,
    "Airport_fee": 1.75,
}
r3 = requests.post(f"{BASE}/predict", json=trip_b)
print("Request 3 — POST /predict (airport ride, weekend evening)")
print(f"  Status: {r3.status_code}  Body: {r3.json()}\n")

# --- Request 4: Batch prediction ---
r4 = requests.post(f"{BASE}/predict/batch", json={"trips": [trip_a, trip_b]})
print("Request 4 — POST /predict/batch (2 trips)")
print(f"  Status: {r4.status_code}  Body: {r4.json()}")

Request 1 — GET /health
  Status: 200  Body: {'status': 'healthy', 'model_loaded': True, 'model_version': '1', 'uptime_seconds': 16.68}

Request 2 — POST /predict (morning commute)
  Status: 200  Body: {'prediction_id': '45318a3f-0bbd-4811-814c-d2c5b8819648', 'tip_amount': 2.56, 'model_version': '1'}

Request 3 — POST /predict (airport ride, weekend evening)
  Status: 200  Body: {'prediction_id': 'b1b5bbe3-d9c5-480b-bb43-b8b8610f4835', 'tip_amount': 10.68, 'model_version': '1'}

Request 4 — POST /predict/batch (2 trips)
  Status: 200  Body: {'predictions': [{'prediction_id': '6688b8f5-b1b1-46be-9e06-1ffbafd9f0eb', 'tip_amount': 2.56, 'model_version': '1'}, {'prediction_id': 'd3cb9410-02c0-4a66-a2d7-b7ab9767d1dd', 'tip_amount': 10.68, 'model_version': '1'}], 'count': 2, 'processing_time_ms': 114.56}


### **<u>Shut down cleanly</u>**

In [32]:
!docker compose down

 Container comp3610-a4-api-1 Stopping 
 Container comp3610-a4-api-1 Stopped 
 Container comp3610-a4-api-1 Removing 
 Container comp3610-a4-api-1 Removed 
 Container comp3610-a4-mlflow-1 Stopping 
 Container comp3610-a4-mlflow-1 Stopped 
 Container comp3610-a4-mlflow-1 Removing 
 Container comp3610-a4-mlflow-1 Removed 
 Network comp3610-a4_default Removing 
 Network comp3610-a4_default Removed 


### <u>**Container size report**</u>

In [33]:
!docker images taxi-tip-api

IMAGE                 ID             DISK USAGE   CONTENT SIZE   EXTRA
taxi-tip-api:latest   b22f06037c48        1.3GB          294MB        


### Configuration Notes

- **Docker Desktop** must be installed and running.
- The API image size is reported above; it is built from `python:3.11-slim` (~150 MB base) plus Python packages and the model artefact.
- Environment variables (`MODEL_PATH`, `MODEL_VERSION`) are set in `docker-compose.yml` so no hard-coded paths exist in the application.
- The MLflow service installs `mlflow==2.22.0` on first start; subsequent starts reuse the cached layer.
- To rebuild after code changes: `docker compose up --build`.

---
### **AI Tools Used**

| Tool | Purpose |
|------|---------|
| **GitHub Copilot** | - Copilot was used for notebook documentation assistance. <br>- Aided in debugging and fixing gaps or errors in code. <br>- Checking against assignment specifications to ensure accuracy and proper implementation. |
| **ChatGPT 5.4** | - Used Chat for recommendations on the structure of the assignment _i.e._ the order in which to start the assignment and which parts should be done first etc. <br>- Documentation assistance for README |